# QLoRA Fine-Tuning: Qwen2.5-1.5B-Instruct

A small Colab proof-of-concept for training LoRA adapters on a 4-bit Qwen model. **The 20-example dataset is for learning the workflow, not production quality.** Use a CUDA GPU (tested on Colab T4).


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes huggingface_hub

In [ ]:
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("All libraries imported successfully!")
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

## Load 4-bit base model

The model is public; no Hugging Face login or token is required for this notebook.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Step 1: Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Step 2: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Step 3: Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

print("4-bit quantized model loaded!")
print("Model quantized:", model.is_loaded_in_4bit)

In [ ]:
import torch
import bitsandbytes as bnb

print("Model:", model.config._name_or_path)
print("4-bit quantized:", model.is_loaded_in_4bit)

# Count 4-bit linear layers
quantized_layers = sum(
    isinstance(layer, bnb.nn.Linear4bit)
    for layer in model.modules()
)

print("4-bit layers:", quantized_layers)

# GPU memory
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

## Baseline inference

Run before attaching adapters. The same generation settings are reused later, but the initial baseline does **not** include the training system prompt, so this is only an informal comparison.

In [ ]:
import torch

def ask_model(question):
    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]

    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return answer


# Test
response = ask_model("What is machine learning?")

print("Model response:")
print(response)

In [ ]:
questions = [
    "What is machine learning?",
    "Explain neural networks to a beginner.",
    "What is an API?",
    "What is gradient descent?",
    "Explain overfitting."
]

baseline_responses = {}

for question in questions:
    answer = ask_model(question)
    baseline_responses[question] = answer

    print("\nQuestion:", question)
    print("Answer:", answer)
    print("-" * 50)

## Prepare QLoRA adapters

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Prepare quantized model for training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Attach LoRA adapters
model = get_peft_model(model, lora_config)

# Show trainable parameters
model.print_trainable_parameters()

## Small prompt-completion dataset

In [ ]:
from datasets import Dataset

examples = [
    ("What is machine learning?",
     "Machine learning helps computers learn patterns from data. Example: predicting house prices."),

    ("What is deep learning?",
     "Deep learning uses neural networks with multiple layers to learn patterns. Example: face recognition."),

    ("What is an API?",
     "An API lets software applications communicate. Example: a weather app fetching forecasts."),

    ("What is Python?",
     "Python is a programming language used for AI, automation and web development. Example: building a chatbot."),

    ("What is overfitting?",
     "Overfitting happens when a model memorizes training data but performs poorly on new data."),

    ("What is gradient descent?",
     "Gradient descent updates model parameters to reduce loss. Think of walking downhill toward the lowest point."),

    ("What is a neural network?",
     "A neural network learns patterns using connected layers of artificial neurons. Example: image classification."),

    ("What is an LLM?",
     "An LLM is a language model trained on large amounts of text. Example: an AI assistant answering questions."),

    ("What is fine-tuning?",
     "Fine-tuning adapts a pretrained model using new training examples. Example: teaching an assistant a specific response style."),

    ("What is LoRA?",
     "LoRA trains small adapter matrices while keeping the original model weights frozen."),

    ("What is QLoRA?",
     "QLoRA combines 4-bit quantization with trainable LoRA adapters to reduce GPU memory usage."),

    ("What is an embedding?",
     "An embedding converts information into a numerical vector. Example: representing similar sentences with nearby vectors."),

    ("What is RAG?",
     "RAG retrieves relevant information before generating an answer. Example: answering questions from PDFs."),

    ("What is classification?",
     "Classification predicts categories. Example: detecting whether an email is spam."),

    ("What is regression?",
     "Regression predicts numerical values. Example: estimating house prices."),

    ("What is supervised learning?",
     "Supervised learning trains models using labeled data. Example: classifying images of cats and dogs."),

    ("What is unsupervised learning?",
     "Unsupervised learning finds patterns in unlabeled data. Example: grouping similar customers."),

    ("What is reinforcement learning?",
     "Reinforcement learning trains an agent through rewards and penalties. Example: learning to play a game."),

    ("What is a transformer?",
     "A transformer uses attention mechanisms to process relationships between tokens. Example: modern language models."),

    ("What is tokenization?",
     "Tokenization splits text into smaller units called tokens so a language model can process it.")
]

system_prompt = (
    "You are a beginner-friendly AI tutor. "
    "Answer in 1-3 short sentences using simple English. "
    "Include one practical example when useful."
)

data = [
    {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        "completion": [
            {"role": "assistant", "content": answer}
        ]
    }
    for question, answer in examples
]

dataset = Dataset.from_list(data).train_test_split(
    test_size=0.2,
    seed=42
)

print(dataset)
print(dataset["train"][0])

## Tokenize and mask prompt tokens

Only the assistant completion contributes to the training loss. Padding and prompt tokens receive label `-100`.

In [ ]:
def tokenize_example(example):
    prompt = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False, add_generation_prompt=True
    )
    full_text = prompt + example["completion"][0]["content"] + tokenizer.eos_token
    encoded = tokenizer(
        full_text, truncation=True, max_length=256, padding="max_length"
    )
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    labels = encoded["input_ids"].copy()
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100
    labels = [
        -100 if mask == 0 else label
        for label, mask in zip(labels, encoded["attention_mask"])
    ]
    encoded["labels"] = labels
    return encoded


tokenized_dataset = dataset.map(
    tokenize_example, remove_columns=["prompt", "completion"]
)
assert all(any(label != -100 for label in row["labels"]) for row in tokenized_dataset["train"])
print(tokenized_dataset)


## Train with standard Transformers Trainer

This notebook uses `Trainer` rather than TRL `SFTTrainer` to avoid a fused-loss incompatibility observed in the original Colab environment. FP16/BF16 automatic mixed precision is disabled because the original run encountered a gradient-scaler error with BF16 adapter gradients.

In [ ]:
from transformers import Trainer, TrainingArguments
import torch

# Check LoRA parameter data types
for name, param in model.named_parameters():
    if param.requires_grad:
        print("Trainable dtype:", param.dtype)
        break

# Create fresh training configuration
args = TrainingArguments(
    output_dir="./qwen-qlora-tutor",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="epoch",
    fp16=False,
    bf16=False,
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False
)

# Create a completely new Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)
print("Starting training...")

trainer.train()

## Preliminary comparison

These questions overlap with training examples and do not control for the system prompt. Do not interpret them as a rigorous benchmark.

In [ ]:
model.eval()

test_questions = [
    "What is machine learning?",
    "Explain neural networks to a beginner.",
    "What is an API?",
    "What is gradient descent?",
    "Explain overfitting."
]

for question in test_questions:
    print("\nQUESTION:", question)
    print("BEFORE:", baseline_responses[question][:300])
    print("AFTER:", ask_model(question))
    print("=" * 60)

## Save LoRA adapters and tokenizer

In [ ]:
# Save trained LoRA adapters
model.save_pretrained("./qwen-qlora-tutor-final")

# Save tokenizer
tokenizer.save_pretrained("./qwen-qlora-tutor-final")

print("QLoRA adapters saved successfully!")

## Optional: copy saved adapters to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

shutil.copytree(
    "./qwen-qlora-tutor-final",
    "/content/drive/MyDrive/qwen-qlora-tutor-final",
    dirs_exist_ok=True
)

print("Saved to Google Drive!")

## Recorded training results

The original Colab run completed **12 steps in 50.75 seconds** over **3 epochs**.

| Epoch | Training loss | Validation loss |
|---|---:|---:|
| 1 | 1.673355 | 1.435578 |
| 2 | 1.155806 | 1.333263 |
| 3 | 1.128460 | 1.332811 |

Only 16 training and 4 validation examples were used. Lower losses do **not** establish a meaningful improvement in response quality; the informal before/after samples remained verbose.
